# 09 — Nonlinear programs with `nlpsol`

**Objectives.** Encode decision variables, parameters, objective, constraints, bounds, and initial guesses at the low level. The canonical form is $\min_x f(x,p)$ subject to $l_g\le g(x,p)\le u_g$ and $l_x\le x\le u_x$. Bounds are data, not symbolic inequalities.


In [ ]:
import casadi as ca
import numpy as np
x = ca.MX.sym('x', 2); p = ca.MX.sym('p')
objective = (x[0]-p)**2 + (x[1]-1)**2
constraints = ca.vertcat(x[0] + x[1], x[0]**2 + x[1]**2)
nlp = {'x':x, 'p':p, 'f':objective, 'g':constraints}
solver = ca.nlpsol('solver', 'ipopt', nlp, {'ipopt.print_level':0, 'print_time':False})
solution = solver(x0=[0.5,0.5], p=2.0, lbx=[0,0], ubx=[3,3], lbg=[1,-ca.inf], ubg=[ca.inf,4])
x_star = np.asarray(solution['x'].full()).ravel()
print('x*', x_star, 'f*', float(solution['f']), 'lambda_g', solution['lam_g'])
print(solver.stats()['return_status'])
assert solver.stats()['success'] and np.isfinite(x_star).all()


Constraint row order must match `lbg`, `ubg`, and `lam_g`. Multipliers help interpret active constraints and construct KKT residuals. Always check `solver.stats()['success']` and `return_status`; receiving an `x` array alone does not establish success.

**Common mistakes:** using an infeasible initial point without realizing it, unscaled variables, NaN-generating expressions, and treating a local result as globally optimal. **Recap:** formulate once, change numeric parameters and bounds between calls.
